In [1]:
import CommunityModelClass3 as cmc3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

save_directory = "phase_diversity"
isExist = os.path.exists(save_directory)
if not isExist:
    os.makedirs(save_directory)

In [2]:
# functions cell

def Phase_search(N_sim,wk_fine,leakage,output_name):
    inputs["leakage"] = leakage
    results = np.zeros((N_sim,len(wk_fine)))

    for epoch in range(N_sim):
        for i in range(len(wk_fine)):

            print("epoch:", epoch, end="\r")
            inputs["k_0"]=wk_fine[i]

            model=cmc3.CommunityModel(inputs)
            model.Simulate()

            data=model.ReturnSimpsonDiversity()
            dd=np.mean(data)

            results[epoch][i]=dd
     

    mean_results=np.mean(results,axis=0) 
    std_results=np.std(results,axis=0)

    output=np.array([mean_results,std_results])
    store_output=pd.DataFrame(output.T)
    store_output.to_csv(save_directory+"/"+output_name,index=False,header=False)
    return mean_results

#----------------------------------------------------------------------------------------------

def sigmoid(x, L ,x0, k, b):
    y = L / (1 + np.exp(-k*(x-x0))) + b
    return y

#----------------------------------------------------------------------------------------------

def Sigmoid_fit(wk_fine,mean_results,output_name):
    from scipy.optimize import curve_fit

    p0 = [max(mean_results), np.median(wk_fine), 1, min(mean_results)] 
    popt, pcov = curve_fit(sigmoid, wk_fine, mean_results, p0, maxfev=int(1e5))

    params = pd.DataFrame(popt)
    variance = pd.DataFrame(pcov)

    save_directory = "phase_diversity"
    params.to_csv(save_directory+"/fit_params_"+output_name,header=False,index=False)
    variance.to_csv(save_directory+"/fit_var_"+output_name,header=False,index=False)

    fit=sigmoid(wk_fine,*popt)
    residuals= mean_results - fit

    return np.array([fit,residuals])

#----------------------------------------------------------------------------------------------

def Fit_plots(wk_fine,mean_results,fit,residuals,std_results,leakage):
    fig, ax = plt.subplots(1,2,figsize=(12,4))
    fig.suptitle("Leakage={}".format(leakage))

    ax[0].set(title="Sigmoid Fit")
    ax[0].scatter(wk_fine,mean_results,marker=".",label="Simulated Data")
    ax[0].plot(wk_fine,fit,ls="dashed",color="firebrick",label="Sigmoid Fit")
    ax[0].set_xscale("log")
    ax[0].set_xlabel("$w_0k_0$")
    ax[0].set_ylabel("Mean Richness")
    ax[0].grid(alpha=0.5)
    ax[0].legend();

    ax[1].set(title="Residuals")
    ax[1].errorbar(wk_fine,residuals,yerr=std_results,fmt=".")
    ax[1].plot(wk_fine,np.zeros_like(wk_fine),ls="--",c="firebrick")
    ax[1].set_xscale("log")
    ax[1].set_xlabel("$w_0k_0$")
    ax[1].grid(alpha=0.5)

#----------------------------------------------------------------------------------------------

def CriticalPoint(fit,wk_fine,popt):
    a = np.array([fit[i] - fit[i - 1] for i in range(1,len(fit))])
    cp_x = wk_fine[1:][a == np.max(a)][0]
    cp_y = sigmoid(cp_x,*popt)
    return [cp_x,cp_y]


In [3]:
inputs={}

inputs["S_tot"] = 80
inputs["S"] = 40
inputs["R"] = 40

# initial conditions
inputs["S0"] = 10
inputs["R0"] = 5

# time simulation params
inputs["time_period"] = 1e5
inputs["t_step"] = 0.1

# partitions params
inputs["num_F"] = 4
inputs["num_T"] = 4
inputs["partition_F"] = [20]*4
inputs["partition_T"] = [10]*4

# activation function
inputs["sigma_type"] = "linear"
inputs["k"] = 20                           
inputs["N"] = 2

# h function params
inputs["tau_R"] = 1
inputs["k_alpha_index"] = 0

# metabolic matrix params
inputs["f_c"] = 0.25
inputs["f_s"] = 0.25
inputs["d0"] = 0.2

# preference matrix params
inputs["preference_type"] = "binary"
inputs["mu_c"] = 4.5
inputs["sigma_c"] = 0.3
inputs["c_0"] = 1
inputs["c_1"] = 1
inputs["q_A"] = 0.5


# save params:
df_inputs=pd.DataFrame(inputs)
df_inputs.to_csv(save_directory+"/simulation_params_phase_diveristy.csv")

In [22]:
N_sim = 10

wk_fine=np.logspace(0, 3, num=30, base=10)
leakage = 0.8

output_name = "phase_diversity_sim10_l08.csv"

rr = Phase_search(N_sim,wk_fine,leakage,output_name)


 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.5555066924880D+03   r2 =  0.1552891618567D-13
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.5555066924880D+03   r2 =  0.1552891618567D-13
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.5555066924880D+03   r2 =  0.1552891618567D-13
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.5555066924880D+03   r2 =  0.1552891618567D-13
 lsoda--  warning..internal t (=r1) 

/Users/raffaelegaudio/opt/anaconda3/lib/python3.9/site-packages/scipy/integrate/_odepack_py.py:247: ODEintWarning: Excess work done on this call (perhaps wrong Dfun type). Run with full_output = 1 to get quantitative information.
  warnings.warn(warning_msg, ODEintWarning)
/Users/raffaelegaudio/Models_Living_Systems/Project_PMLS/CommunityModelClass3.py:484: RuntimeWarning: invalid value encountered in true_divide
  squared_ratio[i,:] = pow(J_in[i] / J_in_total[i], 2)


In [23]:
rr

array([1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.05681253, 1.4428104 , 2.03070482, 1.8147286 ,
       2.70540005, 3.16410175, 3.50336161,        nan, 4.17035321,
       4.49288516, 4.80451476, 4.75270301, 5.16811502, 4.83263037,
       5.00934763, 5.02442753, 5.28707133, 5.60914456, 5.2318274 ,
       5.63661927, 5.06420816, 5.36091287, 4.72585869, 4.78612822])

In [10]:
results=np.genfromtxt(save_directory+"/"+output_name,delimiter=",")
mean_results=results[:,0]
std_results=results[:,1]

mean_results

array([1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.10413162, 1.34396062, 1.88560304, 2.27662471,
       2.6120438 ,        nan, 3.80191856, 3.82175295,        nan,
       4.85753994, 4.31029973, 4.49451681, 4.93953396, 4.75417966,
       4.92944195, 5.26786737, 5.55681236, 5.28809707, 4.96507572,
       5.33990868, 5.18015234, 5.35489813, 5.48959052,        nan])

In [5]:
results=np.genfromtxt(save_directory+"/"+output_name,delimiter=",")
mean_results=results[:,0]
std_results=results[:,1]

ff=Sigmoid_fit(wk_fine,mean_results,output_name)
fit = ff[0]
residuals = ff[1]

Fit_plots(wk_fine,mean_results,fit,residuals,std_results,leakage)

ValueError: array must not contain infs or NaNs

In [4]:
results_09 = np.genfromtxt(save_directory+"/phase_diveristy_sim10_l09.csv",delimiter=",")
mean_results_09 = results_09[:,0]
std_results_09 = results_09[:,1]

fit_params_09 = np.genfromtxt(save_directory+"/fit_params_phase_diveristy_sim10_l09.csv",delimiter=",")
fit_09 = sigmoid(wk_fine, *fit_params_09)
residuals_09 = mean_results_09 -fit_09

Fit_plots(wk_fine,mean_results_09,fit_09,residuals_09,std_results_09,"0.9")


OSError: phase_diversity/phase_diveristy_sim10_l09.csv not found.

In [ ]:

fit_params_09 = np.genfromtxt(save_directory+"/fit_params_phase_diversity_sim10_l09.csv",delimiter=",")
fit_09 = sigmoid(wk_fine, *fit_params_09)
cp_09 = CriticalPoint(fit_09,wk_fine,fit_params_09)

fit_params_08 = np.genfromtxt(save_directory+"/fit_params_phase_diversity_sim10_l08.csv",delimiter=",")
fit_08 = sigmoid(wk_fine, *fit_params_08)
cp_08 = CriticalPoint(fit_08,wk_fine,fit_params_08)

#fit_params_07 = np.genfromtxt(save_directory+"/fit_params_phase_diversity_sim10_l07.csv",delimiter=",")
#fit_07 = sigmoid(wk_fine, *fit_params_07)
#cp_07 = CriticalPoint(fit_07,wk_fine,fit_params_07)

#fit_params_06 = np.genfromtxt(save_directory+"/fit_params_phase_diversity_sim10_l06.csv",delimiter=",")
#fit_06 = sigmoid(wk_fine, *fit_params_06)
#cp_06 = CriticalPoint(fit_06,wk_fine,fit_params_06)

#fit_params_05 = np.genfromtxt(save_directory+"/fit_params_phase_diversity_sim10_l05.csv",delimiter=",")
#fit_05 = sigmoid(wk_fine, *fit_params_05)
#vcp_05 = CriticalPoint(fit_05,wk_fine,fit_params_05)

#fit_params_04 = np.genfromtxt(save_directory+"/fit_params_phase_diversity_sim10_l04.csv",delimiter=",")
#vfit_04 = sigmoid(wk_fine, *fit_params_04)
#cp_04 = CriticalPoint(fit_04,wk_fine,fit_params_04)

In [ ]:
cps=np.array([cp_09,cp_08])

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(cps[:,0],cps[:,1],c="firebrick",marker=".",label="critical point")
plt.plot(wk_fine,fit_09,label="$l=0.9$",alpha=0.6)
#plt.plot(wk_fine,fit_08,label="$l=0.8$",alpha=0.6)
#plt.plot(wk_fine,fit_07,label="$l=0.7$",alpha=0.6)
#plt.plot(wk_fine,fit_06,label="$l=0.6$",alpha=0.6)
#plt.plot(wk_fine,fit_05,label="$l=0.5$",alpha=0.6)
#plt.plot(wk_fine,fit_04,label="$l=0.5$",alpha=0.6)
plt.xscale("log")
plt.xlabel("$w_0k_0$"), plt.ylabel("Mean Simpson Diveristy")
plt.grid(alpha=0.5)
plt.legend();

In [ ]:
cp_09